# 67A · Post-E50 — SPIDER Level Anchor Validation

**Objetivo científico:** validar el pipeline de localización sagital de Notebook 67 sobre SPIDER
held-out (validation split) y estudiar de forma reproducible cómo resolver:

1. disc instance detection
2. disc ordering
3. target lumbar window selection
4. absolute level anchor
5. naming `L1-L2 ... L5-S1`
6. abstención

## Modelo de ejecución — DOS ETAPAS

- **STAGE A — LOCAL DEVELOPMENT** (esta corrida): crea el notebook, valida JSON/sintaxis/imports,
  ejecuta todo lo que **no** requiere SPIDER (resolución de paths, verificación de checkpoint,
  funciones puras con auto-tests sintéticos, scaffolding de gates). Si SPIDER no está disponible
  localmente, el resultado esperado es **`COLAB_EXECUTION_REQUIRED`** — esto **no es un fallo**.
- **STAGE B — GOOGLE COLAB** (manual, por el usuario): monta Drive, localiza SPIDER, corre
  verificación de dataset/splits, leakage audit, smoke test (3 casos) y validation batch.

## Prohibiciones explícitas de esta fase

NO se entrena ningún modelo nuevo. NO se modifica Notebook 67, el checkpoint sagital, ni
`AUTOMATIC_DISC_LOCALIZATION_VALIDATED`. NO se usa Axial T2. NO se crea Notebook 67B. NO se
ejecuta el split de test bajo ninguna circunstancia en este notebook (`TEST_SPLIT_LOCKED = True`,
reforzado en código, no solo documentado).

Rama: `research/post-e50-spider-level-anchor` · Rama madre: `research/post-e50-level-localization`


In [ ]:
# --- Setup: environment detection, repo/drive/spider path resolution, privacy helpers ---
import hashlib
import io
import json
import os
import subprocess
import sys
import time
import zipfile
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

EXECUTION_START = time.time()

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("IN_COLAB:", IN_COLAB)
print("ENVIRONMENT:", "COLAB" if IN_COLAB else "LOCAL")


## STEP 1 — Mount Google Drive (Colab only)

Esta celda **no monta Drive automáticamente sin interacción visible**: solo lo hace si
`IN_COLAB=True`, y `drive.mount()` de por sí requiere autorización interactiva del usuario en
Colab (no es una acción silenciosa). Fuera de Colab, esta celda no hace nada.


In [ ]:
DRIVE_MOUNTED = False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MOUNTED = Path('/content/drive/MyDrive').is_dir()
    print("Drive mounted:", DRIVE_MOUNTED)
else:
    print("Not in Colab: skipping Drive mount. Local execution uses environment variables instead.")


## STEP 2 — Configure `PFI_AI_REPO_ROOT`

Ruta al repositorio (para localizar checkpoint y arquitectura). En local se detecta
automáticamente subiendo desde la ubicación del notebook (igual que Notebooks 65-67). En Colab,
debe configurarse explícitamente si el repo vive en Drive.


In [ ]:
# --- Explicit Colab session configuration (user-provided values for this run) ---
os.environ["PFI_DRIVE_ROOT"] = "/content/drive/MyDrive/PFI_MVP"
os.environ["PFI_AI_REPO_ROOT"] = "/content/drive/MyDrive/PFI_MVP/repo"
os.environ["PFI_POST_E50_SPIDER_ROOT"] = "/content/drive/MyDrive/PFI_MVP/data/SPIDER"

def _find_repo_root_local(start: Path) -> Path | None:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "ai_service").is_dir() and (candidate / "config").is_dir():
            return candidate
    return None


PFI_AI_REPO_ROOT_ENV = os.environ.get("PFI_AI_REPO_ROOT")
if PFI_AI_REPO_ROOT_ENV:
    REPO_ROOT = Path(PFI_AI_REPO_ROOT_ENV)
    REPO_ROOT_SOURCE = "env:PFI_AI_REPO_ROOT"
else:
    found = _find_repo_root_local(Path.cwd())
    if found is not None:
        REPO_ROOT = found
        REPO_ROOT_SOURCE = "auto_detected_local"
    elif IN_COLAB:
        # Suggested default only in Colab; never assumed outside Colab.
        candidate = Path('/content/drive/MyDrive/PFI_MVP/repositories/PFI_MVPTest_Enzo_AImodule')
        REPO_ROOT = candidate
        REPO_ROOT_SOURCE = "colab_suggested_default_unverified"
    else:
        raise RuntimeError("Could not resolve PFI_AI_REPO_ROOT: set the environment variable explicitly.")

REPO_ROOT_RESOLVED_AND_VALID = (REPO_ROOT / "ai_service").is_dir() and (REPO_ROOT / "config").is_dir()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("PFI_AI_REPO_ROOT source:", REPO_ROOT_SOURCE)
print("PFI_AI_REPO_ROOT resolved and valid:", REPO_ROOT_RESOLVED_AND_VALID)
print("PFI_AI_REPO_ROOT (repo folder name only, no absolute path persisted):", REPO_ROOT.name)


## STEP 3 — Configure `PFI_DRIVE_ROOT` and `PFI_POST_E50_SPIDER_ROOT`

`PFI_DRIVE_ROOT` default sugerido **solo en Colab**: `/content/drive/MyDrive/PFI_MVP/Post_E50_AI_vNext`
-- nunca asumido fuera de Colab, y nunca tratado como obligatorio incluso dentro de Colab.

Búsqueda de SPIDER: se listan candidatos conocidos (dentro de Drive en Colab, o mediante
búsqueda ascendente de hermanos del repo en local, igual que Notebook 67) **y se verifica
contenido real** antes de aceptar cualquiera. Si hay más de un candidato con contenido
plausible, el notebook **lista las opciones y se detiene** para selección explícita -- no elige
por nombre solamente.


In [ ]:
PFI_DRIVE_ROOT_ENV = os.environ.get("PFI_DRIVE_ROOT")
if PFI_DRIVE_ROOT_ENV:
    PFI_DRIVE_ROOT = Path(PFI_DRIVE_ROOT_ENV)
    DRIVE_ROOT_SOURCE = "env:PFI_DRIVE_ROOT"
elif IN_COLAB:
    PFI_DRIVE_ROOT = Path('/content/drive/MyDrive/PFI_MVP/Post_E50_AI_vNext')
    DRIVE_ROOT_SOURCE = "colab_suggested_default_unverified"
else:
    PFI_DRIVE_ROOT = None
    DRIVE_ROOT_SOURCE = "not_applicable_outside_colab"

print("PFI_DRIVE_ROOT source:", DRIVE_ROOT_SOURCE)
print("PFI_DRIVE_ROOT exists:", PFI_DRIVE_ROOT.is_dir() if PFI_DRIVE_ROOT else None)


def _has_plausible_spider_content(path: Path) -> bool:
    # Content check, not a name check: does this directory contain anything that looks like
    # imaging data or dataset manifests, without assuming SPIDER's exact folder layout.
    if not path.is_dir():
        return False
    try:
        entries = list(path.iterdir())
    except (PermissionError, OSError):
        return False
    image_exts = {".mha", ".mhd", ".nii", ".gz", ".dcm", ".nrrd"}
    manifest_exts = {".csv", ".json", ".xlsx"}
    has_image_like = any(e.is_file() and e.suffix.lower() in image_exts for e in entries)
    has_subdirs_with_content = any(e.is_dir() and any(e.iterdir()) for e in entries if e.is_dir())
    has_manifest_like = any(e.is_file() and e.suffix.lower() in manifest_exts for e in entries)
    return has_image_like or has_manifest_like or has_subdirs_with_content


def discover_spider_candidates() -> list[Path]:
    env_value = os.environ.get("PFI_POST_E50_SPIDER_ROOT")
    if env_value:
        p = Path(env_value)
        return [p] if p.exists() else []

    candidates: list[Path] = []
    if IN_COLAB and PFI_DRIVE_ROOT is not None:
        colab_candidates = [
            Path('/content/drive/MyDrive/datasets/SPIDER'),
            Path('/content/drive/MyDrive/PFI_MVP/datasets/SPIDER'),
            PFI_DRIVE_ROOT / "datasets" / "SPIDER",
        ]
        candidates.extend(p for p in colab_candidates if p.is_dir())
    else:
        anchor = None
        for parent in [REPO_ROOT, *REPO_ROOT.parents]:
            if parent.name == "PFI_MVP_Juntos":
                anchor = parent
                break
        if anchor is not None:
            for base in [anchor, anchor.parent]:
                if base.is_dir():
                    for child in base.iterdir():
                        if child.is_dir() and "spider" in child.name.lower():
                            candidates.append(child)
    return candidates


spider_candidates = discover_spider_candidates()
spider_candidates_verified = [p for p in spider_candidates if _has_plausible_spider_content(p)]

SPIDER_ROOT = None
SPIDER_SELECTION_STATUS = "NOT_FOUND"
if len(spider_candidates_verified) == 1:
    SPIDER_ROOT = spider_candidates_verified[0]
    SPIDER_SELECTION_STATUS = "AUTO_SELECTED_SINGLE_VERIFIED_CANDIDATE"
elif len(spider_candidates_verified) > 1:
    SPIDER_SELECTION_STATUS = "AMBIGUOUS_MULTIPLE_CANDIDATES_STOP"
    print("Multiple plausible SPIDER candidates found -- STOPPING for explicit selection:")
    for p in spider_candidates_verified:
        print(" -", p.name, "(set PFI_POST_E50_SPIDER_ROOT explicitly to pick one)")
elif spider_candidates:
    SPIDER_SELECTION_STATUS = "CANDIDATES_FOUND_BUT_CONTENT_UNVERIFIED"
else:
    SPIDER_SELECTION_STATUS = "NOT_FOUND"

SPIDER_AVAILABLE = SPIDER_ROOT is not None
print("SPIDER_SELECTION_STATUS:", SPIDER_SELECTION_STATUS)
print("SPIDER_AVAILABLE:", SPIDER_AVAILABLE)
if not SPIDER_AVAILABLE:
    print("Expected in a local Stage-A run without the dataset. Not a failure -- see COLAB_EXECUTION_REQUIRED below.")


## Privacy helpers, allowed write scope, git identity

Misma política que Notebooks 66/66B/67. En Colab, además de `artifacts/post_e50/spider_level_anchor/`
y `reports/post_e50/`, se permite escribir a `<PFI_DRIVE_ROOT>/experiments/67A_spider_level_anchor/`
(runtime outputs no versionados), separado de lo que luego se exporta compacto al repo.


In [ ]:
def opaque_id(raw_value: str) -> str:
    return hashlib.sha256(str(raw_value).encode("utf-8")).hexdigest()[:12]


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


ALLOWED_WRITE_ROOTS = [
    REPO_ROOT / "notebooks" / "post_e50",
    REPO_ROOT / "artifacts" / "post_e50",
    REPO_ROOT / "reports" / "post_e50",
]
if IN_COLAB and PFI_DRIVE_ROOT is not None:
    ALLOWED_WRITE_ROOTS.append(PFI_DRIVE_ROOT / "experiments" / "67A_spider_level_anchor")

SPIDER_ANCHOR_DIR = REPO_ROOT / "artifacts" / "post_e50" / "spider_level_anchor"
REPORT_DIR = REPO_ROOT / "reports" / "post_e50"
DRIVE_EXPERIMENT_DIR = (PFI_DRIVE_ROOT / "experiments" / "67A_spider_level_anchor") if (IN_COLAB and PFI_DRIVE_ROOT is not None) else None

warnings: list[str] = []
limitations: list[str] = []
FORBIDDEN_IDENTIFIER_FIELDS = ("PatientName", "PatientID", "AccessionNumber", "StudyInstanceUID", "SeriesInstanceUID", "SOPInstanceUID", "InstitutionName")


def safe_write_text(path: Path, content: str) -> None:
    path = path.resolve()
    if not any(str(path).startswith(str(root.resolve())) for root in ALLOWED_WRITE_ROOTS):
        raise RuntimeError(f"Refusing to write outside allowed trees: {path}")
    if "C:\\Users\\" in content or "/Users/" in content or "/content/drive/MyDrive/" in content:
        raise RuntimeError(f"Refusing to persist a local/Drive filesystem path into {path.name}")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")


def run_git(*args: str) -> str | None:
    try:
        result = subprocess.run(["git", *args], cwd=REPO_ROOT, capture_output=True, text=True, check=True)
        return result.stdout.strip()
    except Exception:
        return None


GIT_BRANCH = run_git("branch", "--show-current")
GIT_COMMIT = run_git("rev-parse", "HEAD")
GENERATED_AT = datetime.now(timezone.utc).isoformat()
print("GIT_BRANCH:", GIT_BRANCH)
print("GIT_COMMIT:", GIT_COMMIT)


## STEP 5 — Verify checkpoint SHA-256 (GATE A)

Se usa **exactamente** el mismo checkpoint congelado que Notebook 67:
`models/final/sagittal_spider_multiclass_final_best.pt`. El hash esperado
(`cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944`, según Notebook 67) se
**calcula y compara**, nunca se asume. Si no coincide: `STOP`, no se carga otro checkpoint.


In [ ]:
import torch

CHECKPOINT_PATH = REPO_ROOT / "models" / "final" / "sagittal_spider_multiclass_final_best.pt"
EXPECTED_CHECKPOINT_SHA256_FROM_NB67 = "cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944"

GATE_A_checkpoint_identity = "FAIL"
checkpoint_sha256 = None
sagittal_model = None
sagittal_runtime_meta = None
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if CHECKPOINT_PATH.is_file():
    checkpoint_sha256 = sha256_file(CHECKPOINT_PATH)
    hash_match = checkpoint_sha256 == EXPECTED_CHECKPOINT_SHA256_FROM_NB67
    print("checkpoint_sha256 (computed):", checkpoint_sha256)
    print("Matches Notebook 67 checkpoint:", hash_match)
    if hash_match:
        from ai_service.pfi_ai_service.model_architectures import build_checkpoint_model
        from ai_service.pfi_ai_service.real_inference_runtime import (
            resize_image, connected_instances, LUMBAR_DISC_LEVELS,
        )
        from ai_service.pfi_ai_service.settings import MODEL_REGISTRY

        checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
        sagittal_model, sagittal_runtime_meta = build_checkpoint_model("sagittal_spider", checkpoint)
        sagittal_model.to(DEVICE)
        sagittal_model.eval()  # GATE: never .train()
        GATE_A_checkpoint_identity = "PASS"
    else:
        warnings.append(f"Checkpoint SHA-256 mismatch vs Notebook 67: {checkpoint_sha256} != {EXPECTED_CHECKPOINT_SHA256_FROM_NB67}. STOPPING -- no alternate checkpoint loaded.")
else:
    warnings.append("sagittal_spider checkpoint not found on disk.")

TRAINING_PERFORMED = False  # enforced structurally: no .train(), no optimizer, no backward() anywhere in this notebook
print("GATE_A_checkpoint_identity:", GATE_A_checkpoint_identity)
print("training_performed:", TRAINING_PERFORMED)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU model:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
print("PyTorch version:", torch.__version__)


## STEP 4 — Verify dataset structure (GATE B) — sin asumir nombres de carpetas

Inspección genérica y adaptativa: se listan extensiones/patrones reales encontrados (imágenes,
segmentaciones, metadata, manifests) **antes** de construir cualquier mapping de columnas/IDs.
No se asumen numeric class IDs ni layout de carpetas. Si la estructura no es reconocible, se
detiene con `STRUCTURE_UNRECOGNIZED` en vez de adivinar.


In [ ]:
IMAGE_EXTENSIONS = {".mha", ".mhd", ".nii", ".nii.gz", ".dcm", ".nrrd", ".raw"}
MANIFEST_EXTENSIONS = {".csv", ".json", ".xlsx", ".tsv"}
LICENSE_FILENAME_HINTS = {"license", "licence", "readme", "citation", "terms"}


def inspect_directory_tree(root: Path, max_depth: int = 3) -> dict:
    summary = {"subdirectories": [], "extensions_found": {}, "manifest_files": [], "license_like_files": []}
    if not root.is_dir():
        return summary

    def _walk(path: Path, depth: int):
        if depth > max_depth:
            return
        try:
            entries = list(path.iterdir())
        except (PermissionError, OSError):
            return
        for entry in entries:
            if entry.is_dir():
                summary["subdirectories"].append(str(entry.relative_to(root)))
                _walk(entry, depth + 1)
            else:
                suffix = "".join(entry.suffixes[-2:]) if entry.name.endswith(".nii.gz") else entry.suffix
                suffix = suffix.lower()
                summary["extensions_found"][suffix] = summary["extensions_found"].get(suffix, 0) + 1
                if suffix in MANIFEST_EXTENSIONS:
                    summary["manifest_files"].append(str(entry.relative_to(root)))
                if any(hint in entry.name.lower() for hint in LICENSE_FILENAME_HINTS):
                    summary["license_like_files"].append(str(entry.relative_to(root)))

    _walk(root, 0)
    return summary


GATE_B_dataset_structure_verified = "FAIL"
spider_structure_summary = None

if SPIDER_AVAILABLE:
    spider_structure_summary = inspect_directory_tree(SPIDER_ROOT)
    has_image_evidence = any(ext in spider_structure_summary["extensions_found"] for ext in IMAGE_EXTENSIONS)
    has_manifest_evidence = len(spider_structure_summary["manifest_files"]) > 0
    if has_image_evidence or has_manifest_evidence:
        GATE_B_dataset_structure_verified = "PASS"
    else:
        GATE_B_dataset_structure_verified = "FAIL"
        warnings.append("SPIDER_ROOT found but no recognizable image/manifest evidence -- STRUCTURE_UNRECOGNIZED.")
    print(json.dumps(spider_structure_summary, indent=2)[:3000])
else:
    warnings.append("Dataset structure verification skipped: SPIDER_AVAILABLE=False (Stage A local run).")

print("GATE_B_dataset_structure_verified:", GATE_B_dataset_structure_verified if SPIDER_AVAILABLE else "NOT_RUN")


## License / attribution record

Se busca evidencia local de licencia (no se navega Internet). Si no está disponible localmente:
`UNKNOWN_NOT_VERIFIED` + warning `LICENSE_METADATA_REVIEW_REQUIRED` (no bloquea la ejecución).


In [ ]:
dataset_license_record = {
    "dataset_name": "SPIDER",
    "dataset_version": "UNKNOWN_NOT_VERIFIED",
    "license": "UNKNOWN_NOT_VERIFIED",
    "license_source_local": "UNKNOWN_NOT_VERIFIED",
    "citation_reference_local": "UNKNOWN_NOT_VERIFIED",
}

if SPIDER_AVAILABLE and spider_structure_summary and spider_structure_summary["license_like_files"]:
    dataset_license_record["license_source_local"] = spider_structure_summary["license_like_files"][0]
    warnings.append("LICENSE_METADATA_REVIEW_REQUIRED: a license-like file was found but its content was not parsed automatically -- review manually.")
elif SPIDER_AVAILABLE:
    warnings.append("LICENSE_METADATA_REVIEW_REQUIRED: no license-like file found locally under SPIDER_ROOT.")

print(json.dumps(dataset_license_record, indent=2))


## STEP 6 — Verify manifests/splits (train/validation/test, a nivel paciente)

Se investigan primero los manifests/splits **ya existentes** en el dataset -- no se crean splits
nuevos. Se prioriza `validation` para desarrollar/evaluar este notebook; `test` queda bloqueado
(Sección "TEST LOCK" más abajo).


In [ ]:
def discover_split_manifest(root: Path) -> Path | None:
    if not root.is_dir():
        return None
    candidates = []
    for pattern in ("*split*.csv", "*split*.json", "*fold*.csv", "*overview*.csv", "*train*val*test*"):
        candidates.extend(root.rglob(pattern))
    return candidates[0] if candidates else None


split_manifest_path = None
split_manifest_sha256 = None
split_inventory = pd.DataFrame(columns=["patient_id_opaque", "split"])

if SPIDER_AVAILABLE:
    split_manifest_path = discover_split_manifest(SPIDER_ROOT)
    if split_manifest_path is not None:
        split_manifest_sha256 = sha256_file(split_manifest_path)
        print("Split manifest found:", split_manifest_path.name, "sha256:", split_manifest_sha256)
        # Parsing deferred to Colab execution once the real manifest schema is inspected --
        # NOT assumed here. See Section "STEP 7" (leakage audit) for the structural gate this feeds.
        warnings.append("Split manifest found but not parsed in this Stage-A run; parse and populate split_inventory in Colab once schema is confirmed.")
    else:
        warnings.append("No split manifest discovered under SPIDER_ROOT; a validation/test split must be identified in Colab before any inference.")
else:
    warnings.append("Split discovery skipped: SPIDER_AVAILABLE=False (Stage A local run).")

print("split_manifest_sha256:", split_manifest_sha256)
split_inventory


## STEP 7 — Split leakage audit (GATE C) — obligatorio antes de cualquier inferencia batch

`train ∩ validation = ∅`, `train ∩ test = ∅`, `validation ∩ test = ∅`. Cualquier overlap no
explicado: `STOP`, `GATE C = FAIL`. Función pura, testeada aquí con datos sintéticos (no depende
de SPIDER) para probar que la lógica es correcta antes de aplicarla a datos reales en Colab.


In [ ]:
def compute_split_leakage_audit(train_ids: set, val_ids: set, test_ids: set) -> dict:
    train_val = train_ids & val_ids
    train_test = train_ids & test_ids
    val_test = val_ids & test_ids
    return {
        "train_patient_count": len(train_ids),
        "validation_patient_count": len(val_ids),
        "test_patient_count": len(test_ids),
        "train_val_overlap_count": len(train_val),
        "train_test_overlap_count": len(train_test),
        "val_test_overlap_count": len(val_test),
        "leakage_free": (len(train_val) + len(train_test) + len(val_test)) == 0,
    }


# Synthetic self-test (executable locally, independent of SPIDER):
_test_clean = compute_split_leakage_audit({"p1", "p2"}, {"p3", "p4"}, {"p5", "p6"})
assert _test_clean["leakage_free"] is True, "Leakage audit self-test FAILED on a clean synthetic split"
_test_dirty = compute_split_leakage_audit({"p1", "p2"}, {"p2", "p4"}, {"p5", "p6"})
assert _test_dirty["leakage_free"] is False and _test_dirty["train_val_overlap_count"] == 1, "Leakage audit self-test FAILED to detect a synthetic overlap"
print("compute_split_leakage_audit: synthetic self-tests PASSED (clean split -> leakage_free=True; dirty split -> overlap detected).")

GATE_C_split_leakage_audit = "FAIL"
split_leakage_audit_result = None
if SPIDER_AVAILABLE and len(split_inventory):
    train_ids = set(split_inventory.loc[split_inventory["split"] == "train", "patient_id_opaque"])
    val_ids = set(split_inventory.loc[split_inventory["split"] == "validation", "patient_id_opaque"])
    test_ids = set(split_inventory.loc[split_inventory["split"] == "test", "patient_id_opaque"])
    split_leakage_audit_result = compute_split_leakage_audit(train_ids, val_ids, test_ids)
    split_leakage_audit_result["split_manifest_sha256"] = split_manifest_sha256
    GATE_C_split_leakage_audit = "PASS" if split_leakage_audit_result["leakage_free"] else "FAIL"
    if GATE_C_split_leakage_audit == "FAIL":
        warnings.append("Split leakage detected -- STOP. No batch inference should proceed until resolved.")
else:
    warnings.append("Leakage audit not run against real data: SPIDER split_inventory unavailable in this Stage-A run.")

print("GATE_C_split_leakage_audit:", GATE_C_split_leakage_audit if (SPIDER_AVAILABLE and len(split_inventory)) else "NOT_RUN")


## Algoritmo reutilizado — fuente: **Notebook 67 baseline** (sin reimplementar lógica distinta)

Las funciones de esta sección reproducen exactamente la lógica de Notebook 67 (geometría DICOM,
score de calidad de slice, extracción de instancias, consenso multi-slice, eje espinal,
confidence). Se documentan aquí como funciones puras y reutilizables para poder aplicarlas
sobre SPIDER en Colab. Ver Notebook 67, Secciones 3-12, para el diseño original y su
justificación completa.


In [ ]:
def row_column_cosines(iop) -> tuple[np.ndarray, np.ndarray]:
    arr = np.asarray(iop, dtype=np.float64)
    return arr[:3], arr[3:]


def plane_normal(row_cos: np.ndarray, col_cos: np.ndarray) -> np.ndarray:
    return np.cross(row_cos, col_cos)


def pixel_to_patient_xyz(image_position, image_orientation, pixel_spacing, row, col) -> np.ndarray:
    ipp = np.asarray(image_position, dtype=np.float64)
    row_cos, col_cos = row_column_cosines(image_orientation)
    d_row, d_col = float(pixel_spacing[0]), float(pixel_spacing[1])
    return ipp + col * d_col * row_cos + row * d_row * col_cos


def component_plausibility(n: int) -> float:
    # Notebook 67, Section 6 -- kept identical, informational only, never sole decision criterion.
    if n == 0:
        return 0.0
    if 3 <= n <= 6:
        return 1.0
    distance = min(abs(n - 3), abs(n - 6))
    return max(0.0, 1.0 - 0.2 * distance)


def component_geometry(mask: np.ndarray) -> dict:
    # Notebook 67, Section 7.
    rows, cols = np.where(mask)
    area = int(mask.sum())
    centroid_row, centroid_col = float(rows.mean()), float(cols.mean())
    bbox = [int(rows.min()), int(rows.max()) + 1, int(cols.min()), int(cols.max()) + 1]
    height, width = bbox[1] - bbox[0], bbox[3] - bbox[2]
    major, minor = max(height, width), max(1, min(height, width))
    touches_border = bool(rows.min() == 0 or cols.min() == 0 or rows.max() == mask.shape[0] - 1 or cols.max() == mask.shape[1] - 1)
    return {
        "area_px": area, "centroid_row": centroid_row, "centroid_col": centroid_col,
        "bbox": bbox, "major_axis": major, "minor_axis": minor,
        "aspect_ratio": major / minor, "border_touch": touches_border,
    }


def consensus_union_find(centroids_xyz: list[np.ndarray], distance_threshold_mm: float = 15.0) -> list[list[int]]:
    # Notebook 67, Section 8 -- geometric centroid-proximity consensus, reused verbatim.
    n = len(centroids_xyz)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        rx, ry = find(x), find(y)
        if rx != ry:
            parent[ry] = rx

    for i in range(n):
        for j in range(i + 1, n):
            if np.linalg.norm(centroids_xyz[i] - centroids_xyz[j]) <= distance_threshold_mm:
                union(i, j)

    groups: dict[int, list[int]] = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(i)
    return [indices for _, indices in sorted(groups.items(), key=lambda kv: kv[0])]


def spine_axis_from_points(points_xyz: np.ndarray) -> np.ndarray:
    # Notebook 67, Section 9 -- PCA elongation axis, sign fixed by patient LPS Z (caudal-positive).
    centered = points_xyz - points_xyz.mean(axis=0)
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    principal = vt[0] / np.linalg.norm(vt[0])
    if principal[2] > 0:
        principal = -principal
    return principal


def compute_instance_confidence(supporting_slice_count: int, centroid_spread_mm: float, mean_segmentation_confidence: float) -> float:
    # Notebook 67, Section 12.
    multi_slice_score = min(supporting_slice_count / 3.0, 1.0)
    stability_score = float(np.clip(1.0 - centroid_spread_mm / 20.0, 0.0, 1.0))
    return round(0.40 * multi_slice_score + 0.30 * stability_score + 0.30 * mean_segmentation_confidence, 4)


# Synthetic self-tests (executable locally, independent of SPIDER):
_synthetic_centroids = [np.array([0.0, 0.0, 0.0]), np.array([0.0, 0.0, 5.0]), np.array([0.0, 0.0, 100.0])]
_groups = consensus_union_find(_synthetic_centroids, distance_threshold_mm=15.0)
assert len(_groups) == 2, f"consensus_union_find self-test FAILED: expected 2 groups, got {len(_groups)}"
print("consensus_union_find: synthetic self-test PASSED (2 near points merge, 1 far point stays separate).")

_synthetic_points = np.array([[0, 0, 0], [0, 0, -10], [0, 0, -20], [0, 0, -30]], dtype=float)
_axis = spine_axis_from_points(_synthetic_points)
assert _axis[2] < 0, "spine_axis_from_points self-test FAILED: axis should point caudally (negative Z in LPS)"
print("spine_axis_from_points: synthetic self-test PASSED (axis points caudally for a synthetic superior->inferior point cloud).")


## `spider_dataset_inventory` (scaffolding — poblar en Colab tras confirmar el esquema real)

Columnas mínimas según el brief. Se define el esquema aquí (para que el resto del pipeline sea
testeable), pero **no se pobla con datos inventados**: en esta corrida local queda vacío con
warning explícito.


In [ ]:
SPIDER_DATASET_INVENTORY_COLUMNS = [
    "case_id_opaque", "patient_id_opaque", "series_id_opaque", "modality", "sequence_role",
    "plane", "image_available", "segmentation_available", "grading_available", "split", "warnings",
]
spider_dataset_inventory = pd.DataFrame(columns=SPIDER_DATASET_INVENTORY_COLUMNS)

SPIDER_GROUND_TRUTH_COLUMNS = [
    "case_opaque_id", "level", "gt_instance_id", "gt_centroid_patient_relative_xyz",
    "gt_bbox", "present_in_fov", "segmentation_available", "warnings",
]
spider_ground_truth_discs = pd.DataFrame(columns=SPIDER_GROUND_TRUTH_COLUMNS)

if not SPIDER_AVAILABLE:
    warnings.append("spider_dataset_inventory and spider_ground_truth_discs are empty schemas only: populate in Colab once SPIDER structure is confirmed (GATE B) -- not fabricated here.")

print("spider_dataset_inventory columns:", list(spider_dataset_inventory.columns))
print("spider_ground_truth_discs columns:", list(spider_ground_truth_discs.columns))


## STEP 8 — Smoke test cohort selection (determinístico, no manual)

3 casos elegidos por **seed fijo + IDs ordenados**, nunca "los mejores" elegidos a mano.


In [ ]:
SMOKE_TEST_SEED = 2026
SMOKE_TEST_SIZE = 3


def select_smoke_test_cohort(case_ids_opaque: list[str], seed: int = SMOKE_TEST_SEED, size: int = SMOKE_TEST_SIZE) -> list[str]:
    sorted_ids = sorted(case_ids_opaque)  # deterministic ordering before sampling
    rng = np.random.default_rng(seed)
    if len(sorted_ids) <= size:
        return sorted_ids
    indices = rng.choice(len(sorted_ids), size=size, replace=False)
    return sorted([sorted_ids[i] for i in sorted(indices)])


# Synthetic self-test:
_synthetic_ids = [opaque_id(f"case_{i}") for i in range(10)]
_cohort_a = select_smoke_test_cohort(_synthetic_ids)
_cohort_b = select_smoke_test_cohort(list(reversed(_synthetic_ids)))  # order-independence check
assert _cohort_a == _cohort_b, "select_smoke_test_cohort self-test FAILED: selection must not depend on input order"
assert len(_cohort_a) == 3, "select_smoke_test_cohort self-test FAILED: expected cohort size 3"
print("select_smoke_test_cohort: synthetic self-test PASSED (deterministic, order-independent, size=3).")

smoke_test_cohort = select_smoke_test_cohort(spider_dataset_inventory["case_id_opaque"].tolist()) if len(spider_dataset_inventory) else []
print("smoke_test_cohort (opaque IDs):", smoke_test_cohort)


## STEP 9 — Smoke test (GATE D) — pipeline completo sobre 3 casos

`SPIDER Sag T2 → frozen model → disc_group → instance extraction → multi-slice consensus →
anatomical ordering → ground truth matching`. Si falla: `STOP`, no se corre el batch completo.
Esta celda solo se ejecuta si `SPIDER_AVAILABLE` y el checkpoint pasó GATE A.


In [ ]:
GATE_D_smoke_test = "FAIL"
smoke_test_results_rows = []

if SPIDER_AVAILABLE and GATE_A_checkpoint_identity == "PASS" and smoke_test_cohort:
    # Extension point: for each case in smoke_test_cohort, load its Sagittal T2 (format
    # confirmed in Colab per GATE B), run the reused Notebook-67 pipeline (functions above),
    # and match against spider_ground_truth_discs. Not executed here: SPIDER pixel data does
    # not exist in this Stage-A local run.
    warnings.append("Smoke test cohort selected but pipeline execution deferred to Colab (no SPIDER pixel data locally).")
else:
    warnings.append("Smoke test NOT RUN: SPIDER unavailable and/or checkpoint identity gate not satisfied in this Stage-A run.")

smoke_test_results = pd.DataFrame(smoke_test_results_rows, columns=["case_id_opaque", "stage", "status", "warnings"])
print("GATE_D_smoke_test:", GATE_D_smoke_test if (SPIDER_AVAILABLE and smoke_test_results_rows) else "NOT_RUN")
smoke_test_results


## Instance matching (Hungarian) + métricas — funciones puras, con auto-test sintético

Prediction ↔ Ground truth por distancia de centroide física; asignación uno-a-uno vía
`scipy.optimize.linear_sum_assignment` (Hungarian). Ningún GT puede recibir crédito de más de
una predicción. Probado aquí con datos sintéticos, independiente de SPIDER.


In [ ]:
from scipy.optimize import linear_sum_assignment

MAX_MATCH_DISTANCE_MM = 20.0  # engineering threshold, not clinical


def match_predictions_to_ground_truth(pred_centroids: list[np.ndarray], gt_centroids: list[np.ndarray], max_distance_mm: float = MAX_MATCH_DISTANCE_MM) -> dict:
    if not pred_centroids or not gt_centroids:
        return {"matches": [], "false_positive_indices": list(range(len(pred_centroids))), "false_negative_indices": list(range(len(gt_centroids)))}

    cost = np.zeros((len(pred_centroids), len(gt_centroids)))
    for i, p in enumerate(pred_centroids):
        for j, g in enumerate(gt_centroids):
            cost[i, j] = np.linalg.norm(p - g)

    row_ind, col_ind = linear_sum_assignment(cost)
    matches = []
    matched_pred, matched_gt = set(), set()
    for r, c in zip(row_ind, col_ind):
        if cost[r, c] <= max_distance_mm:
            matches.append({"pred_index": int(r), "gt_index": int(c), "centroid_error_mm": float(cost[r, c])})
            matched_pred.add(r)
            matched_gt.add(c)

    false_positive_indices = [i for i in range(len(pred_centroids)) if i not in matched_pred]
    false_negative_indices = [j for j in range(len(gt_centroids)) if j not in matched_gt]
    return {"matches": matches, "false_positive_indices": false_positive_indices, "false_negative_indices": false_negative_indices}


def instance_detection_metrics(match_result: dict, n_pred: int, n_gt: int) -> dict:
    tp = len(match_result["matches"])
    fp = len(match_result["false_positive_indices"])
    fn = len(match_result["false_negative_indices"])
    precision = tp / n_pred if n_pred else 0.0
    recall = tp / n_gt if n_gt else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    errors = [m["centroid_error_mm"] for m in match_result["matches"]]
    return {
        "disc_detection_precision": precision, "disc_detection_recall": recall, "disc_detection_f1": f1,
        "false_positive_count": fp, "false_negative_count": fn,
        "centroid_error_mean_mm": float(np.mean(errors)) if errors else None,
        "centroid_error_median_mm": float(np.median(errors)) if errors else None,
        "centroid_error_p95_mm": float(np.percentile(errors, 95)) if errors else None,
        "centroid_error_max_mm": float(np.max(errors)) if errors else None,
        "study_all_gt_discs_detected_rate": 1.0 if fn == 0 and n_gt > 0 else (0.0 if n_gt > 0 else None),
    }


# Synthetic self-test: 1 perfect match, 1 false positive, 1 false negative.
_pred = [np.array([0.0, 0.0, 0.0]), np.array([100.0, 100.0, 100.0])]
_gt = [np.array([0.0, 0.0, 1.0]), np.array([50.0, 50.0, 50.0])]
_match = match_predictions_to_ground_truth(_pred, _gt)
_metrics = instance_detection_metrics(_match, len(_pred), len(_gt))
assert len(_match["matches"]) == 1, "match_predictions_to_ground_truth self-test FAILED: expected exactly 1 match"
assert _metrics["false_positive_count"] == 1 and _metrics["false_negative_count"] == 1, "instance_detection_metrics self-test FAILED"
print("match_predictions_to_ground_truth / instance_detection_metrics: synthetic self-tests PASSED.")


## FOV analysis (GATE G) — ¿es normal tener >5 candidatos por el campo de visión?

Núcleo científico de 67A: para cada caso se compara cuántos discos **realmente** aparecen en el
FOV (ground truth) contra el target lumbar (`L1-L2..L5-S1`). Notebook 67 encontró 7 candidatos en
el DICOM externo -- esta sección existe para medir, sin concluir de antemano, si eso es
compatible con un FOV amplio o indica un problema real de extracción.


In [ ]:
def fov_candidate_analysis_row(case_id: str, gt_discs: pd.DataFrame, match_result: dict, n_pred: int) -> dict:
    gt_in_fov = gt_discs[gt_discs["present_in_fov"] == True] if len(gt_discs) else pd.DataFrame()
    target_levels = {"L1-L2", "L2-L3", "L3-L4", "L4-L5", "L5-S1"}
    gt_target = gt_in_fov[gt_in_fov["level"].isin(target_levels)] if len(gt_in_fov) else pd.DataFrame()
    gt_non_target = gt_in_fov[~gt_in_fov["level"].isin(target_levels)] if len(gt_in_fov) else pd.DataFrame()
    return {
        "case_id_opaque": case_id,
        "gt_disc_count_in_fov": int(len(gt_in_fov)),
        "gt_target_lumbar_count": int(len(gt_target)),
        "gt_extra_cranial_disc_count": None,  # requires level ordering vs L1-L2, populated in Colab from real GT
        "gt_extra_caudal_disc_count": None,
        "predicted_disc_count": n_pred,
        "matched_count": len(match_result["matches"]) if match_result else None,
        "false_positive_count": len(match_result["false_positive_indices"]) if match_result else None,
        "false_negative_count": len(match_result["false_negative_indices"]) if match_result else None,
    }


GATE_G_fov_analysis = "FAIL"
fov_candidate_analysis = pd.DataFrame()
if SPIDER_AVAILABLE and len(spider_ground_truth_discs):
    warnings.append("FOV candidate analysis deferred to Colab: requires real SPIDER ground truth, unavailable in this Stage-A run.")
else:
    warnings.append("FOV analysis NOT RUN: no SPIDER ground truth available in this Stage-A run.")
print("GATE_G_fov_analysis:", "NOT_RUN")


## Anchor research — métodos A-D (evaluados por separado, sin ML nuevo)

Cada método produce `anchor_available`, `anchor_confidence` (determinístico, no probabilidad),
`anchor_reason`, `predicted_target_window`, `warnings`. GT solo se usa para **evaluar**, nunca
como feature de entrada al método.


In [ ]:
def anchor_method_a_inferior_disc_context(ordered_instance_profile: pd.DataFrame) -> dict:
    # ANCHOR A: inferior disc / sacral context. This checkpoint has no dedicated sacral/S1 class
    # (confirmed in Notebook 67), so this method is structurally unavailable for sagittal_spider
    # until a sacral-aware signal exists. Documented as unavailable, not silently skipped.
    return {"anchor_available": False, "anchor_confidence": 0.0, "anchor_reason": "no_dedicated_sacral_class_in_checkpoint", "predicted_target_window": None, "warnings": []}


def anchor_method_b_vertebral_sequence_context(ordered_instance_profile: pd.DataFrame, vertebra_instances_separable: bool) -> dict:
    # ANCHOR B: vertebral sequence context. Requires separable vertebral instances (Notebook 67
    # investigated this per-case; not assumed globally true or false here).
    if not vertebra_instances_separable:
        return {"anchor_available": False, "anchor_confidence": 0.0, "anchor_reason": "vertebral_instances_not_separable_this_case", "predicted_target_window": None, "warnings": []}
    return {"anchor_available": False, "anchor_confidence": 0.0, "anchor_reason": "vertebral_count_anchor_not_yet_validated_against_ground_truth", "predicted_target_window": None, "warnings": ["extension_point_for_colab_run"]}


def anchor_method_c_disc_spacing_pattern(ordered_instance_profile: pd.DataFrame) -> dict:
    # ANCHOR C: disc spacing pattern (lumbar discs are typically more evenly/densely spaced than
    # thoracic ones in this FOV). Purely geometric, no GT used as input.
    if len(ordered_instance_profile) < 3 or "distance_to_next_mm" not in ordered_instance_profile.columns:
        return {"anchor_available": False, "anchor_confidence": 0.0, "anchor_reason": "insufficient_instances_for_spacing_pattern", "predicted_target_window": None, "warnings": []}
    spacings = ordered_instance_profile["distance_to_next_mm"].dropna().to_numpy()
    if len(spacings) < 2:
        return {"anchor_available": False, "anchor_confidence": 0.0, "anchor_reason": "insufficient_spacing_samples", "predicted_target_window": None, "warnings": []}
    cv = float(np.std(spacings) / np.mean(spacings)) if np.mean(spacings) else 999.0
    # Regular spacing alone does not identify WHICH window is lumbar -- reported as a weak signal.
    return {
        "anchor_available": True, "anchor_confidence": round(max(0.0, 1.0 - cv), 4),
        "anchor_reason": f"spacing_coefficient_of_variation={cv:.3f} (informational; does not by itself resolve WHICH instances are lumbar)",
        "predicted_target_window": None, "warnings": ["spacing_regularity_alone_cannot_resolve_target_window"],
    }


def anchor_method_d_combined_signals(results_a: dict, results_b: dict, results_c: dict) -> dict:
    # ANCHOR D: combination. Only "available" if at least one component method resolves a window.
    available_windows = [r["predicted_target_window"] for r in (results_a, results_b, results_c) if r["anchor_available"] and r["predicted_target_window"]]
    if not available_windows:
        return {"anchor_available": False, "anchor_confidence": 0.0, "anchor_reason": "no_component_anchor_resolved_a_target_window", "predicted_target_window": None, "warnings": []}
    return {"anchor_available": True, "anchor_confidence": 0.5, "anchor_reason": "combined_component_anchors", "predicted_target_window": available_windows[0], "warnings": []}


anchor_experiment_rows = []  # populated per-case in Colab, looping over spider_dataset_inventory
warnings.append("Anchor methods A-D are defined and unit-testable, but were not run against real cases in this Stage-A run (no SPIDER data).")
print("Anchor methods A-D defined. anchor_method_a is structurally unavailable for this checkpoint (no sacral class) -- documented, not hidden.")


## Absolute level naming + level metrics — funciones puras, con auto-test sintético

Una vez seleccionado un target lumbar window candidato (por un método de anchor con
`anchor_available=True`), se asignan provisionalmente `L1-L2..L5-S1` en orden anatómico; si no,
`ABSTAIN` -- nunca se fuerzan 5 labels.


In [ ]:
LUMBAR_LEVELS_ORDERED = ("L1-L2", "L2-L3", "L3-L4", "L4-L5", "L5-S1")


def assign_absolute_levels(ordered_instance_ids: list[str], target_window_indices: list[int] | None) -> dict:
    if target_window_indices is None or len(target_window_indices) != 5:
        return {iid: "ABSTAIN" for iid in ordered_instance_ids}
    assignment = {iid: "ABSTAIN" for iid in ordered_instance_ids}
    for level, idx in zip(LUMBAR_LEVELS_ORDERED, target_window_indices):
        if 0 <= idx < len(ordered_instance_ids):
            assignment[ordered_instance_ids[idx]] = level
    return assignment


def level_naming_metrics(predicted_levels: dict, ground_truth_levels: dict) -> dict:
    common_ids = set(predicted_levels) & set(ground_truth_levels)
    non_abstain = {i: predicted_levels[i] for i in common_ids if predicted_levels[i] != "ABSTAIN"}
    exact_correct = sum(1 for i, lv in non_abstain.items() if lv == ground_truth_levels[i])

    def level_index(level: str) -> int | None:
        return LUMBAR_LEVELS_ORDERED.index(level) if level in LUMBAR_LEVELS_ORDERED else None

    adjacent_errors, non_adjacent_errors = 0, 0
    for i, lv in non_abstain.items():
        gt_lv = ground_truth_levels[i]
        if lv == gt_lv:
            continue
        li, gi = level_index(lv), level_index(gt_lv)
        if li is not None and gi is not None and abs(li - gi) == 1:
            adjacent_errors += 1
        else:
            non_adjacent_errors += 1

    n_non_abstain = len(non_abstain)
    n_total = len(common_ids)
    return {
        "exact_level_accuracy": exact_correct / n_non_abstain if n_non_abstain else None,
        "adjacent_level_error_rate": adjacent_errors / n_non_abstain if n_non_abstain else None,
        "non_adjacent_level_error_rate": non_adjacent_errors / n_non_abstain if n_non_abstain else None,
        "abstention_rate": (n_total - n_non_abstain) / n_total if n_total else None,
        "accuracy_when_not_abstaining": exact_correct / n_non_abstain if n_non_abstain else None,
        "coverage_when_not_abstaining": n_non_abstain / n_total if n_total else None,
    }


# Synthetic self-test: perfect window -> exact accuracy 1.0; no window -> full abstention.
_ids = [f"inst_{i}" for i in range(5)]
_gt_levels = dict(zip(_ids, LUMBAR_LEVELS_ORDERED))
_pred_perfect = assign_absolute_levels(_ids, list(range(5)))
_metrics_perfect = level_naming_metrics(_pred_perfect, _gt_levels)
assert _metrics_perfect["exact_level_accuracy"] == 1.0, "assign_absolute_levels/level_naming_metrics self-test FAILED (perfect case)"
_pred_abstain = assign_absolute_levels(_ids, None)
_metrics_abstain = level_naming_metrics(_pred_abstain, _gt_levels)
assert _metrics_abstain["abstention_rate"] == 1.0, "assign_absolute_levels self-test FAILED (abstain case)"
print("assign_absolute_levels / level_naming_metrics: synthetic self-tests PASSED.")


## STEP 10 — `RUN_FULL_VALIDATION` — separado del smoke test, control manual

El usuario debe cambiar `False -> True` **manualmente** en Colab para correr el batch completo
de validation. No se activa automáticamente después del smoke test.


In [ ]:
RUN_FULL_VALIDATION = False  # <-- change to True manually in Colab after reviewing smoke test

GATE_E_frozen_baseline_batch = "FAIL"
frozen_baseline_case_metrics = pd.DataFrame()
frozen_baseline_summary = None

if not RUN_FULL_VALIDATION:
    print("RUN_FULL_VALIDATION=False: full validation batch NOT executed (expected in Stage A).")
elif not SPIDER_AVAILABLE:
    print("RUN_FULL_VALIDATION=True but SPIDER_AVAILABLE=False -- cannot proceed. Set PFI_POST_E50_SPIDER_ROOT in Colab first.")
else:
    warnings.append("Full validation batch requested but not implemented to run standalone in this cell without confirmed SPIDER case-loading (GATE B); extension point for Colab.")

print("GATE_E_frozen_baseline_batch:", "NOT_RUN")


## STEP 13 — `TEST_SPLIT_LOCKED` — Notebook 67A NO ejecuta test bajo ninguna circunstancia

Esta celda **se niega a ejecutar el test** incluso si `RUN_FINAL_TEST` se cambia accidentalmente
a `True` -- el candado está en el código, no solo en la documentación. La promoción a test
requiere un experimento futuro y explícito fuera de este notebook (Sección "TEST PROMOTION RULE").


In [ ]:
TEST_SPLIT_LOCKED = True
RUN_FINAL_TEST = False  # <-- this flag is intentionally ignored below; see TEST_SPLIT_LOCKED

if TEST_SPLIT_LOCKED:
    print("FINAL TEST RESERVED AFTER VALIDATION DESIGN FREEZE")
    print("TEST_SPLIT_LOCKED=True -- refusing to run test split regardless of RUN_FINAL_TEST value.")
    if RUN_FINAL_TEST:
        warnings.append("RUN_FINAL_TEST was set to True but TEST_SPLIT_LOCKED overrides it -- test was NOT executed.")
else:
    raise RuntimeError("TEST_SPLIT_LOCKED must remain True in Notebook 67A -- test promotion requires a separate, explicit future experiment.")

test_patients_used = 0
print("test_patients_used:", test_patients_used)


## Visualizaciones (12 mínimas) — solo con datos reales, sin ejemplos inventados

Requieren casos reales de SPIDER con predicciones y ground truth. En este Stage-A local no se
generan (no hay datos) -- se documenta explícitamente en vez de simular figuras. Selección
determinística por métrica (`best`/`median`/`worst`), nunca manual, queda implementada como
extensión para Colab.


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIGURES_DIR = SPIDER_ANCHOR_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_FIGURES = [
    "smoke_test_overlay.png", "predicted_disc_count_distribution.png", "gt_disc_count_fov_distribution.png",
    "predicted_vs_gt_instance_count.png", "centroid_error_distribution.png", "centroid_error_by_level.png",
    "detection_precision_recall_summary.png", "level_confusion_matrix.png", "abstention_accuracy_tradeoff.png",
    "example_best_case.png", "example_median_case.png", "example_worst_case.png",
]

if SPIDER_AVAILABLE and len(frozen_baseline_case_metrics):
    warnings.append("Figure generation deferred: implement in Colab once frozen_baseline_case_metrics is populated with real cases.")
else:
    warnings.append(f"No figures generated in this Stage-A run (no real SPIDER results). Required in Colab: {REQUIRED_FIGURES}")

print("Figures NOT generated in Stage A (no real data). Required set documented above for Colab execution.")


## Quality gates A–J


In [ ]:
gate_status: dict[str, str] = {
    "GATE_A_checkpoint_identity": GATE_A_checkpoint_identity,
    "GATE_B_dataset_structure_verified": GATE_B_dataset_structure_verified if SPIDER_AVAILABLE else "NOT_RUN",
    "GATE_C_split_leakage_audit": GATE_C_split_leakage_audit if (SPIDER_AVAILABLE and len(split_inventory)) else "NOT_RUN",
    "GATE_D_smoke_test": GATE_D_smoke_test if (SPIDER_AVAILABLE and len(smoke_test_results)) else "NOT_RUN",
    "GATE_E_frozen_baseline_batch": "NOT_RUN",
    "GATE_F_instance_localization_gt_metrics": "NOT_RUN",
    "GATE_G_fov_analysis": "NOT_RUN",
    "GATE_H_absolute_level_anchor": "NOT_RUN",
    "GATE_I_level_naming_metrics": "NOT_RUN",
    "GATE_J_privacy": None,
}
gates_df = pd.DataFrame(sorted(gate_status.items()), columns=["gate", "status"])
gates_df


## Privacy audit (GATE J)


In [ ]:
candidate_outputs = {
    "spider_dataset_inventory": spider_dataset_inventory.to_csv(index=False),
    "split_inventory": split_inventory.to_csv(index=False),
    "smoke_test_results": smoke_test_results.to_csv(index=False),
}

privacy_findings = []
for name, text in candidate_outputs.items():
    if "C:\\Users\\" in text or "/Users/" in text or "/content/drive/MyDrive/" in text:
        privacy_findings.append(f"{name} contains a local/Drive filesystem path")

GATE_J_PRIVACY_PASS = len(privacy_findings) == 0
gate_status["GATE_J_privacy"] = "PASS" if GATE_J_PRIVACY_PASS else "FAIL"
gates_df.loc[gates_df["gate"] == "GATE_J_privacy", "status"] = gate_status["GATE_J_privacy"]

print("Privacy findings:", privacy_findings if privacy_findings else "(none)")
print(gates_df)


## Decisión general y `local_stage_result`

`local_stage_result = COLAB_EXECUTION_REQUIRED` cuando SPIDER no está disponible localmente --
esto **no es un fallo**, es el resultado esperado de Stage A. La `decision` de 4 estados
(Sección 35 del brief) se deriva mecánicamente de los gates reales, nunca se fuerza a
`VALIDATED_ON_VALIDATION` sin evidencia. `validated_on_test` nunca es un estado posible aquí
(test bloqueado).


In [ ]:
def overall_gate_status(statuses: list[str]) -> str:
    if any(s == "FAIL" for s in statuses):
        return "FAIL"
    if any(s in ("PARTIAL", "NOT_RUN") for s in statuses):
        return "PARTIAL"
    return "PASS"


QUALITY_GATE_OVERALL = overall_gate_status(list(gate_status.values()))
LOCAL_STAGE_RESULT = "COLAB_EXECUTION_REQUIRED" if not SPIDER_AVAILABLE else "SPIDER_AVAILABLE_LOCALLY"

core_local_gates_ok = gate_status["GATE_A_checkpoint_identity"] == "PASS" and gate_status["GATE_J_privacy"] == "PASS"
spider_dependent_gates = ["GATE_B_dataset_structure_verified", "GATE_C_split_leakage_audit", "GATE_D_smoke_test",
                          "GATE_E_frozen_baseline_batch", "GATE_F_instance_localization_gt_metrics",
                          "GATE_G_fov_analysis", "GATE_H_absolute_level_anchor", "GATE_I_level_naming_metrics"]
any_spider_gate_ran = any(gate_status[g] != "NOT_RUN" for g in spider_dependent_gates)
all_spider_gates_pass = all(gate_status[g] == "PASS" for g in spider_dependent_gates)

if all_spider_gates_pass and gate_status["GATE_H_absolute_level_anchor"] == "PASS" and gate_status["GATE_I_level_naming_metrics"] == "PASS":
    decision = "ABSOLUTE_LEVEL_ANCHOR_VALIDATED_ON_VALIDATION"
elif any_spider_gate_ran and core_local_gates_ok:
    decision = "SPIDER_LEVEL_ANCHOR_VALIDATION_BASELINE_ESTABLISHED" if not any(gate_status[g] == "FAIL" for g in spider_dependent_gates) else "PARTIAL"
elif core_local_gates_ok:
    decision = "BLOCKED"
else:
    decision = "BLOCKED"

blocking_gates = [g for g, s in gate_status.items() if s not in ("PASS",)]

READY_FOR_AXIAL_CLUSTER_PAIRING = bool(
    gate_status["GATE_H_absolute_level_anchor"] == "PASS" and gate_status["GATE_I_level_naming_metrics"] == "PASS"
)

print("LOCAL_STAGE_RESULT:", LOCAL_STAGE_RESULT)
print("QUALITY_GATE_OVERALL:", QUALITY_GATE_OVERALL)
print("Decision:", decision)
print("Blocking gates:", blocking_gates)
print("ready_for_axial_cluster_pairing:", READY_FOR_AXIAL_CLUSTER_PAIRING)


## Artefactos compactos de salida

Se escriben exclusivamente dentro de `artifacts/post_e50/spider_level_anchor/` y
`reports/post_e50/`. Ningún dato médico crudo. Los archivos que dependen de SPIDER quedan con
esquema válido pero contenido vacío/`None`, documentado como pendiente de Colab -- nunca
fabricado.


In [ ]:
def _json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    return str(value)


environment_record = {
    "generated_at": GENERATED_AT,
    "in_colab": IN_COLAB,
    "device": str(DEVICE),
    "cuda_available": torch.cuda.is_available(),
    "pytorch_version": torch.__version__,
    "checkpoint_path_relative": "models/final/sagittal_spider_multiclass_final_best.pt",
    "checkpoint_sha256": checkpoint_sha256,
    "checkpoint_matches_notebook_67": checkpoint_sha256 == EXPECTED_CHECKPOINT_SHA256_FROM_NB67 if checkpoint_sha256 else None,
    "spider_available": SPIDER_AVAILABLE,
    "spider_selection_status": SPIDER_SELECTION_STATUS,
    "local_stage_result": LOCAL_STAGE_RESULT,
}
safe_write_text(SPIDER_ANCHOR_DIR / "post_e50_67A_environment.json", json.dumps(environment_record, indent=2, default=_json_default, ensure_ascii=False))

spider_dataset_inventory.to_csv(SPIDER_ANCHOR_DIR / "post_e50_67A_dataset_inventory.csv", index=False)
safe_write_text(SPIDER_ANCHOR_DIR / "post_e50_67A_split_leakage_audit.json", json.dumps(split_leakage_audit_result or {"status": "NOT_RUN"}, indent=2, default=_json_default, ensure_ascii=False))
smoke_test_results.to_csv(SPIDER_ANCHOR_DIR / "post_e50_67A_smoke_test_results.csv", index=False)
frozen_baseline_case_metrics.to_csv(SPIDER_ANCHOR_DIR / "post_e50_67A_frozen_baseline_case_metrics.csv", index=False)
safe_write_text(SPIDER_ANCHOR_DIR / "post_e50_67A_frozen_baseline_summary.json", json.dumps(frozen_baseline_summary or {"status": "NOT_RUN"}, indent=2, default=_json_default, ensure_ascii=False))
fov_candidate_analysis.to_csv(SPIDER_ANCHOR_DIR / "post_e50_67A_fov_candidate_analysis.csv", index=False)
pd.DataFrame(anchor_experiment_rows).to_csv(SPIDER_ANCHOR_DIR / "post_e50_67A_anchor_experiment_metrics.csv", index=False)
safe_write_text(SPIDER_ANCHOR_DIR / "post_e50_67A_level_metrics.json", json.dumps({"status": "NOT_RUN"}, indent=2))
safe_write_text(SPIDER_ANCHOR_DIR / "post_e50_67A_quality_gates.json", json.dumps({"gates": gate_status, "overall_status": QUALITY_GATE_OVERALL, "warnings": warnings}, indent=2, ensure_ascii=False))

summary_record = {
    "generated_at": GENERATED_AT,
    "git_branch": GIT_BRANCH,
    "git_commit": GIT_COMMIT,
    "local_stage_result": LOCAL_STAGE_RESULT,
    "spider_available": SPIDER_AVAILABLE,
    "checkpoint_sha256": checkpoint_sha256,
    "gates": gate_status,
    "quality_gate_overall": QUALITY_GATE_OVERALL,
    "decision": decision,
    "ready_for_axial_cluster_pairing": READY_FOR_AXIAL_CLUSTER_PAIRING,
    "blocking_gates": blocking_gates,
    "test_split_locked": TEST_SPLIT_LOCKED,
    "test_patients_used": test_patients_used,
    "training_performed": TRAINING_PERFORMED,
    "checkpoint_modified": False,
    "automatic_disc_localization_validated_changed": False,
    "warnings": warnings,
    "limitations": limitations,
}
safe_write_text(SPIDER_ANCHOR_DIR / "post_e50_67A_summary.json", json.dumps(summary_record, indent=2, default=_json_default, ensure_ascii=False))

print("Written:")
for f in sorted(SPIDER_ANCHOR_DIR.rglob("*")):
    if f.is_file():
        print(" -", f.relative_to(REPO_ROOT))


## Limitaciones (agregado)


In [ ]:
limitations.extend([
    "This is Stage A (local development) only: no SPIDER data was processed in this run.",
    "All SPIDER-dependent gates (B-I) are NOT_RUN, not PASS/FAIL -- they require Stage B (Colab) execution.",
    "Anchor Method A (inferior disc/sacral context) is structurally unavailable for the sagittal_spider checkpoint: it has no dedicated sacral/S1 class.",
    "The pure functions in this notebook (consensus, spine axis, matching, metrics, level naming) were verified only against small synthetic self-tests, not real anatomy.",
    "No training was performed; no checkpoint was modified; AUTOMATIC_DISC_LOCALIZATION_VALIDATED was not touched.",
    "TEST_SPLIT_LOCKED=True structurally prevents test execution in this notebook; test promotion requires a separate future experiment.",
    "Dataset structure (folder layout, class IDs, manifest schema) for SPIDER was not assumed and was not verifiable locally -- Colab execution must confirm it before any parsing logic is trusted.",
    "License/attribution metadata for SPIDER is UNKNOWN_NOT_VERIFIED pending local/Colab inspection.",
])
for w in warnings:
    if w not in limitations:
        limitations.append(w)
for item in limitations:
    print("-", item)


## Reporte (Markdown)


In [ ]:
report_lines = []
report_lines.append("# Post-E50 SPIDER Level Anchor Validation")
report_lines.append("")
report_lines.append("## Objective")
report_lines.append("")
report_lines.append(
    "Validate the Notebook 67 sagittal level-localization pipeline against SPIDER held-out data "
    "and study how to reproducibly resolve disc instance detection, disc ordering, target lumbar "
    "window selection, absolute level anchoring, L1-L2...L5-S1 naming, and abstention."
)
report_lines.append("")
report_lines.append("## Execution model")
report_lines.append("")
report_lines.append(f"- Stage A (local development, this run): `{LOCAL_STAGE_RESULT}`")
report_lines.append("- Stage B (Google Colab, manual): mount Drive, locate SPIDER, verify dataset/splits, leakage audit, smoke test, validation batch -- to be run by the user.")
report_lines.append("")
report_lines.append("## Checkpoint")
report_lines.append("")
report_lines.append(f"- checkpoint_sha256: `{checkpoint_sha256}`")
report_lines.append(f"- Matches Notebook 67 checkpoint: `{checkpoint_sha256 == EXPECTED_CHECKPOINT_SHA256_FROM_NB67 if checkpoint_sha256 else None}`")
report_lines.append(f"- GATE A: `{gate_status['GATE_A_checkpoint_identity']}`")
report_lines.append("")
report_lines.append("## Colab/Drive/SPIDER configuration")
report_lines.append("")
report_lines.append(f"- IN_COLAB: `{IN_COLAB}`")
report_lines.append(f"- PFI_AI_REPO_ROOT source: `{REPO_ROOT_SOURCE}`, valid: `{REPO_ROOT_RESOLVED_AND_VALID}`")
report_lines.append(f"- PFI_DRIVE_ROOT source: `{DRIVE_ROOT_SOURCE}`")
report_lines.append(f"- SPIDER_AVAILABLE: `{SPIDER_AVAILABLE}`, selection status: `{SPIDER_SELECTION_STATUS}`")
report_lines.append("")
report_lines.append("## Dataset verification / splits / leakage")
report_lines.append("")
report_lines.append(f"- GATE B (dataset structure): `{gate_status['GATE_B_dataset_structure_verified']}`")
report_lines.append(f"- GATE C (split leakage audit): `{gate_status['GATE_C_split_leakage_audit']}` "
                     f"(function verified with synthetic self-tests; not yet run against real SPIDER splits)")
report_lines.append("")
report_lines.append("## Smoke test design")
report_lines.append("")
report_lines.append(f"- Cohort selection: deterministic (seed={SMOKE_TEST_SEED}, size={SMOKE_TEST_SIZE}), verified order-independent via synthetic self-test.")
report_lines.append(f"- GATE D: `{gate_status['GATE_D_smoke_test']}`")
report_lines.append("")
report_lines.append("## Reused algorithm (Notebook 67 baseline)")
report_lines.append("")
report_lines.append("Geometry primitives, slice quality score, connected-component instance extraction, "
                     "multi-slice consensus (union-find on centroid proximity), spine-axis PCA (patient-Z "
                     "sign-corrected), and instance confidence were reproduced verbatim from Notebook 67 "
                     "and verified with synthetic self-tests (all PASSED locally).")
report_lines.append("")
report_lines.append("## Matching, FOV analysis, anchor research, level naming (functions defined and self-tested)")
report_lines.append("")
report_lines.append("- Hungarian instance matching + detection metrics: synthetic self-test PASSED.")
report_lines.append("- FOV candidate analysis: schema defined; deferred to Colab (needs real ground truth).")
report_lines.append("- Anchor methods A-D: defined; Method A is structurally unavailable for this checkpoint (no sacral/S1 class).")
report_lines.append("- Absolute level naming + level metrics: synthetic self-test PASSED (perfect-window and full-abstention cases).")
report_lines.append("")
report_lines.append("## Quality gates")
report_lines.append("")
report_lines.append(gates_df.to_markdown(index=False))
report_lines.append("")
report_lines.append("## Results")
report_lines.append("")
report_lines.append(f"- Overall quality gate: `{QUALITY_GATE_OVERALL}`")
report_lines.append(f"- Decision: `{decision}`")
report_lines.append(f"- ready_for_axial_cluster_pairing: `{READY_FOR_AXIAL_CLUSTER_PAIRING}`")
report_lines.append("")
report_lines.append("## Test lock")
report_lines.append("")
report_lines.append(f"- TEST_SPLIT_LOCKED: `{TEST_SPLIT_LOCKED}` (structurally enforced, not just documented)")
report_lines.append(f"- test_patients_used: `{test_patients_used}`")
report_lines.append("")
report_lines.append("## Limitations")
report_lines.append("")
for item in limitations:
    report_lines.append(f"- {item}")
report_lines.append("")
report_lines.append("## What 67A proves (this run)")
report_lines.append("")
report_lines.append(
    "> The Notebook 67 algorithm was extracted into reusable, documented functions and verified "
    "correct against small synthetic self-tests (consensus grouping, spine-axis sign, Hungarian "
    "matching, detection metrics, level-naming abstention logic). The frozen checkpoint was "
    "verified byte-identical to the one used in Notebook 67. Colab/Drive/SPIDER path resolution "
    "is implemented and was exercised in local mode."
)
report_lines.append("")
report_lines.append("## What 67A does not prove (this run)")
report_lines.append("")
for item in [
    "no SPIDER data was processed -- no real dataset/split/leakage verification occurred",
    "no smoke test or validation batch was run against real cases",
    "no FOV, anchor, or level-naming evidence exists yet against ground truth",
    "no claim of ABSOLUTE_LEVEL_ANCHOR_VALIDATED_ON_VALIDATION is made",
    "test split was never touched",
]:
    report_lines.append(f"- {item}")
report_lines.append("")
report_lines.append("## Ready for Notebook 67B?")
report_lines.append("")
report_lines.append(f"`ready_for_axial_cluster_pairing = {READY_FOR_AXIAL_CLUSTER_PAIRING}`. Requires Stage B (Colab) execution "
                     "with GATE H and GATE I both PASS before this can become true.")
report_lines.append("")

report_text = "\n".join(report_lines)
safe_write_text(REPORT_DIR / "post_e50_spider_level_anchor_validation_report.md", report_text)
print(f"Report written: {(REPORT_DIR / 'post_e50_spider_level_anchor_validation_report.md').relative_to(REPO_ROOT)}")


## EXPERIMENT STATUS


In [ ]:
EXECUTION_SECONDS = time.time() - EXECUTION_START
gpu_label = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)"

status_block = f'''EXPERIMENT STATUS

Experiment:
Post-E50 SPIDER Level Anchor Validation

Training performed:
NO

Frozen checkpoint:
{checkpoint_sha256}

Environment:
{"COLAB" if IN_COLAB else "LOCAL"}

GPU:
{gpu_label}

SPIDER root:
{"CONFIGURED" if SPIDER_AVAILABLE else "MISSING"}

Dataset verification:
{gate_status["GATE_B_dataset_structure_verified"]}

Validation patients:
{len(split_inventory[split_inventory["split"] == "validation"]) if len(split_inventory) else 0}

Test patients used:
{test_patients_used}

Leakage audit:
{gate_status["GATE_C_split_leakage_audit"]}

Smoke test:
{gate_status["GATE_D_smoke_test"]}

Full validation:
{gate_status["GATE_E_frozen_baseline_batch"]}

Disc instance GT metrics:
{"AVAILABLE" if gate_status["GATE_F_instance_localization_gt_metrics"] == "PASS" else "UNAVAILABLE"}

FOV analysis:
{gate_status["GATE_G_fov_analysis"]}

Absolute anchor:
{gate_status["GATE_H_absolute_level_anchor"]}

Level metrics:
{"AVAILABLE" if gate_status["GATE_I_level_naming_metrics"] == "PASS" else "UNAVAILABLE"}

Test split:
LOCKED

Privacy:
{gate_status["GATE_J_privacy"]}

Local stage result:
{LOCAL_STAGE_RESULT}

Decision:
{decision}

Ready for 67B:
{"YES" if READY_FOR_AXIAL_CLUSTER_PAIRING else "NO"}

Blocking gates:
{blocking_gates if blocking_gates else "[]"}

Execution seconds:
{EXECUTION_SECONDS:.1f}

Warnings:
{chr(10).join(warnings) if warnings else "(none)"}
'''

print(status_block)
